# L23 demo: an eval gate that fails CI, and drift you can measure before it costs you

Two runnable exercises. **Part A** builds and tests a CI eval gate exactly the
way L21's harness would be wired into a pipeline: a script that reads a frozen
metric, compares it against a threshold, and exits non-zero when the system has
regressed, plus a real, syntax-checked GitHub Actions workflow that would run it
on every push. **Part B** measures data drift on real L3/L4 Intel Lab sensor
data with PSI and the Kolmogorov-Smirnov test, three ways: no drift at all, the
real gradual drift a mote's own battery produces over its deployment, and a
synthetic sudden shift standing in for the module's "retrofitted compressor"
scenario.

No hosted LLM, no live GitHub Actions run: this sandbox cannot trigger a real
CI job, so the workflow file below is validated for correctness (it parses as
YAML and its logic is unit-tested locally) rather than watched turning red in
an actual GitHub UI. Push it to a real repository, per L21/A11, to see that.

## Part A: an eval gate

### The gate itself

A frozen threshold per metric, and a rule for which direction is bad: too low
for something like faithfulness, too high for something like MAE. This is
deliberately the smallest possible version of "gate on a metric threshold,"
built to be read in thirty seconds.

In [ ]:
import json
import sys

# direction=False means "lower is worse" (fail if value < threshold);
# direction=True means "higher is worse" (fail if value > threshold).
GATES = {
    'faithfulness_rate':   {'threshold': 0.90, 'higher_is_worse': False},
    'reference_match_rate': {'threshold': 0.70, 'higher_is_worse': False},
    'mae_good_slice':      {'threshold': 3.00, 'higher_is_worse': True},
}

def evaluate_gate(metrics: dict) -> list:
    """Return a list of human-readable failure strings; empty means the gate passed."""
    failures = []
    for name, rule in GATES.items():
        if name not in metrics:
            continue
        value = metrics[name]
        if rule['higher_is_worse'] and value > rule['threshold']:
            failures.append(f"{name}={value} exceeds allowed maximum {rule['threshold']}")
        elif not rule['higher_is_worse'] and value < rule['threshold']:
            failures.append(f"{name}={value} below required minimum {rule['threshold']}")
    return failures

def run_gate(metrics: dict) -> int:
    failures = evaluate_gate(metrics)
    if failures:
        print('EVAL GATE FAILED:')
        for f in failures:
            print(' -', f)
        return 1
    print('eval gate passed:', metrics)
    return 0


### Testing the gate before trusting it with your merge button

The gate is a plain function, so it gets a plain unit test, no CI system
required to know it works. Two cases: L21's actual measured numbers (should
pass), and the same numbers with the RAG faithfulness rate regressed by a
prompt change that started dropping citations (should fail, for a named
reason).

In [ ]:
# L21's real, measured numbers
good_run = {'faithfulness_rate': 1.00, 'reference_match_rate': 0.75, 'mae_good_slice': 2.49}
assert run_gate(good_run) == 0

print()
# a regression: a prompt change stops citing sources reliably
regressed_run = {'faithfulness_rate': 0.65, 'reference_match_rate': 0.75, 'mae_good_slice': 2.49}
assert run_gate(regressed_run) == 1

print()
# a surrogate that has quietly gotten worse
worse_surrogate = {'faithfulness_rate': 1.00, 'reference_match_rate': 0.75, 'mae_good_slice': 4.10}
assert run_gate(worse_surrogate) == 1

print()
print('all gate behaviors verified')


### The pipeline this gate belongs to

Lint and unit tests first, because a syntax error should never waste eval-harness
time. The eval harness runs against the frozen set (L21) and produces
`metrics.json`. The gate above reads it. Only on a pass does the pipeline build a
deployable container. This file is checked for valid YAML and has the same
job structure a real workflow would; it is not wired into this course
repository's own CI, since this repository builds a book, not an ML system.

In [ ]:
import yaml

WORKFLOW_YAML = '''\
name: eval-gate
on: [push, pull_request]

jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Install uv
        uses: astral-sh/setup-uv@v3

      - name: Install dependencies
        run: uv sync --frozen

      - name: Lint and unit tests
        run: uv run pytest tests/ -x

      - name: Run the eval harness on the frozen set
        run: uv run python -m pipeline.eval --eval-set eval_set.jsonl --out metrics.json

      - name: Gate on the metric threshold
        run: uv run python eval_gate.py metrics.json

      - name: Log this run's metrics to MLflow
        run: uv run python -m pipeline.log_run --metrics metrics.json
        if: always()

  build:
    needs: eval
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build the deployable container
        run: docker build -t myapp:${{ github.sha }} .
'''

parsed = yaml.safe_load(WORKFLOW_YAML)
assert parsed['jobs']['build']['needs'] == 'eval'   # the container never builds if the gate step failed
print('workflow YAML is valid;', len(parsed['jobs']), 'jobs, build depends on eval:', parsed['jobs']['build']['needs'])


`needs: eval` is the line doing the real work in that file. The `build` job,
the one that produces something deployable, cannot start until the `eval` job
finishes successfully, and `eval` finishes unsuccessfully the moment
`eval_gate.py` exits non-zero. Push a change that regresses `metrics.json`
below the frozen threshold, and this pipeline never reaches the step that
would have shipped it.

## Part B: drift, measured before it costs you

Same Intel Lab Parquet file as L3, L4, L19, and L21. **PSI** (Population
Stability Index) bins a reference distribution and compares how the
production distribution's mass has moved between those same bins; a common
rule of thumb reads PSI under 0.1 as no meaningful shift, 0.1 to 0.25 as
worth watching, and above 0.25 as a real shift. The **Kolmogorov-Smirnov
test** compares two samples' empirical distributions directly and gives a
p-value for "these could plausibly be the same distribution."

Three comparisons, in order of how much drift they should show.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)
PARQUET = CACHE / 'readings.parquet'
URL = 'https://raw.githubusercontent.com/linsea423/Intel_Lab_Data/master/data.zip'
COLS = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']

if not PARQUET.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        with z.open('data.txt') as f:
            raw = pd.read_csv(f, sep=r'\s+', header=None, names=COLS, on_bad_lines='skip')
    df = raw.dropna(subset=['moteid']).copy()
    df['moteid'] = pd.to_numeric(df['moteid'], errors='coerce')
    df = df.dropna(subset=['moteid'])
    df['moteid'] = df['moteid'].astype(int)
    df = df[df.moteid.between(1, 54)]
    df['ts'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='mixed', errors='coerce')
    df = df.dropna(subset=['ts'])
    for c in ['temperature', 'humidity', 'light', 'voltage']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['temperature', 'humidity', 'light', 'voltage'])
    df[['moteid', 'ts', 'temperature', 'humidity', 'light', 'voltage']].sort_values(
        ['moteid', 'ts']).to_parquet(PARQUET, index=False)

df = pd.read_parquet(PARQUET)
df = df[(df.temperature.between(0, 50)) & (df.voltage.between(1.5, 3.0))]
mote1 = df[df.moteid == 1].sort_values('ts')
print(len(mote1), 'readings from mote 1')


In [ ]:
def psi(reference, production, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference, bins=edges)
    prod_counts, _ = np.histogram(production, bins=edges)
    ref_pct = np.clip(ref_counts / len(reference), 1e-4, None)
    prod_pct = np.clip(prod_counts / len(production), 1e-4, None)
    return float(np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct)))

def drift_report(reference, production, feature):
    p = psi(reference, production)
    k = stats.ks_2samp(reference, production)
    flag = 'ALERT' if p > 0.25 else ('watch' if p > 0.10 else 'ok')
    print(f'{feature:12s} PSI={p:6.3f} [{flag:5s}]  KS stat={k.statistic:.3f}  KS p={k.pvalue:.2e}')


### Comparison 1: no drift at all

A pure sanity check. Randomly split the same mote's readings into two halves;
there is no reason to expect any distributional difference, and the metrics
should say so.

In [ ]:
rng_split = mote1.sample(frac=1.0, random_state=0)
half = len(rng_split) // 2
random_a, random_b = rng_split.iloc[:half], rng_split.iloc[half:]

for feat in ['temperature', 'voltage']:
    drift_report(random_a[feat].to_numpy(), random_b[feat].to_numpy(), feat)


### Comparison 2: real, gradual drift, from the mote's own deployment

First week of this mote's readings as the reference distribution, last week as
"production." No injection here at all, this is what the real sensor actually
did over roughly a month in the lab.

In [ ]:
first_week = mote1[mote1.ts < mote1.ts.min() + pd.Timedelta(days=7)]
last_week = mote1[mote1.ts > mote1.ts.max() - pd.Timedelta(days=7)]

for feat in ['temperature', 'voltage']:
    drift_report(first_week[feat].to_numpy(), last_week[feat].to_numpy(), feat)


Voltage alerts hard, and it should: a battery discharging over a month is
exactly the kind of slow, real, physically-caused drift the notes call
**concept drift when it changes what a reading means** (a "low" voltage
reading in week four is normal aging, not the fault code it would be in week
one) and **data drift** in the plain distributional sense measured here. The
temperature shift is real too, milder, and consistent with the lab's
environment warming through the deployment window (Intel Lab's data runs
February to April 2004). Neither of these needed anything injected to show
up; they are what the sensor was actually doing, which is the entire point of
monitoring input distributions instead of assuming your training-time
snapshot stays true.

### Comparison 3: a sudden shift, standing in for a retrofit

The module's scenario: a compressor gets retrofitted and its operating
envelope moves. Simulate the sensor equivalent, a batch of readings from a
recalibrated or replaced sensor, by injecting a shift into a copy of
otherwise-ordinary data instead of waiting a month for it to happen for
real.

In [ ]:
rng = np.random.default_rng(0)
retrofit_batch = random_b.copy()
retrofit_batch['voltage'] = retrofit_batch['voltage'] * 1.15 + rng.normal(0, 0.05, len(retrofit_batch))

drift_report(random_a['voltage'].to_numpy(), retrofit_batch['voltage'].to_numpy(), 'voltage (retrofit)')


### Reading the three results together

| Comparison | What it represents | PSI |
|---|---|---|
| Random split | no drift | ~0.0 |
| First week vs. last week | real, gradual, explainable drift | large |
| Injected shift | a sudden regime change | large |

PSI alone cannot tell you *why* a distribution moved, only that it did. Both
the real gradual case and the injected sudden case alert; telling them apart,
"the battery is aging on schedule" versus "something changed overnight,"
needs a human looking at the trend, not a single threshold crossing. That
distinction, and the false-alarm-versus-missed-drift trade-off in choosing
the threshold at all, is exactly what the notes argue a rolling baseline over
a static one is for.

Full notes, with the failure-mode/guardrail table and the responsible-AI
material this notebook does not cover: [`../notes.md`](notes.md).